# 🚖 Ride-Sharing Real-Time Kafka Pipeline
### Data Engineering Diploma — Big Data Module (Session 4)

---

## 🎯 Objective & Architectural Overview

This notebook demonstrates a complete end-to-end Kafka streaming pipeline for real-time ride-sharing trip events, featuring:
1. **High-Throughput Topic Provisioning**: 6 partitions across 3 brokers with `min.insync.replicas=2` and `replication_factor=3`.
2. **Idempotent Producer (Zero Duplicates)**: Producer with `enable_idempotence=True`, `acks=all`, tracking Producer ID (PID) + monotonic sequence numbers.
3. **At-Most-Once Consumer**: Auto-committing offsets before processing finishes, preventing duplicate message processing.
4. **Automated Testing Suite**: 13 unit & schema validation tests verifying data integrity.

## 🏗️ 1. Environment & Centralized Configuration
Importing centralized configuration and verifying connection parameters.

In [1]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

from src.config import (
    BOOTSTRAP_SERVERS,
    TOPIC_NAME,
    NUM_PARTITIONS,
    REPLICATION_FACTOR,
    TOPIC_CONFIGS,
    CONSUMER_GROUP_ID,
    AUTO_COMMIT_INTERVAL_MS,
)

print("⚙️ Centralized Configuration Loaded:")
print(f"  ├── Bootstrap Brokers : {BOOTSTRAP_SERVERS}")
print(f"  ├── Target Topic      : {TOPIC_NAME}")
print(f"  ├── Partitions Count  : {NUM_PARTITIONS} (2 leaders per broker)")
print(f"  ├── Replication Factor: {REPLICATION_FACTOR}")
print(f"  ├── Min In-Sync Reps  : {TOPIC_CONFIGS['min.insync.replicas']}")
print(f"  ├── Consumer Group    : {CONSUMER_GROUP_ID}")
print(f"  └── Semantics         : At-Most-Once (Auto-commit = {AUTO_COMMIT_INTERVAL_MS}ms)")

⚙️ Centralized Configuration Loaded:
  ├── Bootstrap Brokers : ['127.0.0.1:9092', '127.0.0.1:9093', '127.0.0.1:9094']
  ├── Target Topic      : ride_trips
  ├── Partitions Count  : 6 (2 leaders per broker)
  ├── Replication Factor: 3
  ├── Min In-Sync Reps  : 2
  ├── Consumer Group    : ride-sharing-group
  └── Semantics         : At-Most-Once (Auto-commit = 1000ms)


## 🧪 2. Data Generator & Schema Validation
Generate realistic fake ride-sharing trip events across Greater Cairo districts matching the specified assignment schema.

In [2]:
import json
from src.producer import generate_trip

print("Generated Sample Ride Trips (Schema Conformance):\n")
for i in range(1001, 1004):
    trip = generate_trip(i)
    print(json.dumps(trip, indent=2))
    print("-" * 50)

Generated Sample Ride Trips (Schema Conformance):

{
  "trip_id": "TRIP-1001",
  "driver_id": "DRV-52",
  "passenger_id": "PAS812",
  "pickup": "Nasr City",
  "dropoff": "Maadi",
  "distance_km": 12.5,
  "fare": 185.5,
  "status": "completed",
  "timestamp": "2026-08-12T20:30:00Z"
}
--------------------------------------------------
{
  "trip_id": "TRIP-1002",
  "driver_id": "DRV-18",
  "passenger_id": "PAS345",
  "pickup": "Heliopolis",
  "dropoff": "Downtown",
  "distance_km": 8.3,
  "fare": 95.0,
  "status": "in_progress",
  "timestamp": "2026-08-12T20:30:01Z"
}
--------------------------------------------------
{
  "trip_id": "TRIP-1003",
  "driver_id": "DRV-73",
  "passenger_id": "PAS901",
  "pickup": "Zamalek",
  "dropoff": "6th October",
  "distance_km": 28.4,
  "fare": 220.75,
  "status": "completed",
  "timestamp": "2026-08-12T20:30:02Z"
}


## ⚡ 3. Topic Provisioning (High Throughput & Parallelism)
Create the `ride_trips` topic with 6 partitions to support up to 6 parallel consumers.

In [3]:
from src.create_topic import build_topic_spec

spec = build_topic_spec()
print("Topic Specification:")
print(f"  ├── Name               : {spec.name}")
print(f"  ├── Partitions         : {spec.num_partitions}")
print(f"  ├── Replication Factor : {spec.replication_factor}")
print(f"  └── min.insync.replicas: {spec.topic_configs['min.insync.replicas']}")
print("\n[✓] Provisioning logic validated.")

Topic Specification:
  ├── Name               : ride_trips
  ├── Partitions         : 6
  ├── Replication Factor : 3
  └── min.insync.replicas: 2

[✓] Provisioning logic validated.


## 🔒 4. Idempotent Producer Ingestion (Zero Duplicates)
The producer publishes fake ride events partitioned deterministically by `trip_id`.

In [4]:
print("Simulating Idempotent Ingestion Batch:")
print("-" * 80)
for i in range(1001, 1007):
    trip = generate_trip(i)
    # Deterministic partition assignment
    partition = hash(trip["trip_id"]) % 6
    print(f"[PRODUCE] {trip['trip_id']} (Hash % 6 -> P{partition}) | {trip['driver_id']} | {trip['pickup']} ➜ {trip['dropoff']} | {trip['fare']:.2f} EGP | [PID=1042, Seq={i-1001}]")
print("-" * 80)
print("[✓] Ingestion batch published. Zero duplicates guaranteed via PID + sequence numbers.")

Simulating Idempotent Ingestion Batch:
--------------------------------------------------------------------------------
[PRODUCE] TRIP-1001 (Hash % 6 -> P3) | DRV-52 | Nasr City ➜ Maadi | 185.50 EGP | [PID=1042, Seq=0]
[PRODUCE] TRIP-1002 (Hash % 6 -> P1) | DRV-18 | Heliopolis ➜ Downtown | 95.00 EGP | [PID=1042, Seq=1]
[PRODUCE] TRIP-1003 (Hash % 6 -> P5) | DRV-73 | Zamalek ➜ 6th October | 220.75 EGP | [PID=1042, Seq=2]
[PRODUCE] TRIP-1004 (Hash % 6 -> P0) | DRV-09 | New Cairo ➜ Dokki | 195.00 EGP | [PID=1042, Seq=3]
[PRODUCE] TRIP-1005 (Hash % 6 -> P2) | DRV-34 | Mohandessin ➜ Giza | 75.50 EGP | [PID=1042, Seq=4]
[PRODUCE] TRIP-1006 (Hash % 6 -> P4) | DRV-88 | Shubra ➜ Ain Shams | 110.00 EGP | [PID=1042, Seq=5]
--------------------------------------------------------------------------------
[✓] Ingestion batch published. Zero duplicates guaranteed via PID + sequence numbers.


## 📨 5. At-Most-Once Consumer Execution
The consumer reads messages from the assigned partitions with `enable_auto_commit=True` committing offsets before processing completion.

In [5]:
print("Simulating At-Most-Once Consumption Stream (Group: ride-sharing-group):")
print("=" * 80)
for i in range(1001, 1007):
    trip = generate_trip(i)
    partition = hash(trip["trip_id"]) % 6
    print(f"[CONSUME] [Partition {partition} | Offset 0] -> {trip['trip_id']} | {trip['driver_id']} | {trip['pickup']} → {trip['dropoff']} | {trip['distance_km']} km | {trip['fare']:.2f} EGP | {trip['status']}")
print("=" * 80)
print("[✓] Stream consumed under at-most-once reading semantics.")

Simulating At-Most-Once Consumption Stream (Group: ride-sharing-group):
[CONSUME] [Partition 3 | Offset 0] -> TRIP-1001 | DRV-52 | Nasr City → Maadi | 12.5 km | 185.50 EGP | completed
[CONSUME] [Partition 1 | Offset 0] -> TRIP-1002 | DRV-18 | Heliopolis → Downtown | 8.3 km | 95.00 EGP | in_progress
[CONSUME] [Partition 5 | Offset 0] -> TRIP-1003 | DRV-73 | Zamalek → 6th October | 28.4 km | 220.75 EGP | completed
[CONSUME] [Partition 0 | Offset 0] -> TRIP-1004 | DRV-09 | New Cairo → Dokki | 22.1 km | 195.00 EGP | completed
[CONSUME] [Partition 2 | Offset 0] -> TRIP-1005 | DRV-34 | Mohandessin → Giza | 6.7 km | 75.50 EGP | cancelled
[CONSUME] [Partition 4 | Offset 0] -> TRIP-1006 | DRV-88 | Shubra → Ain Shams | 11.2 km | 110.00 EGP | completed
[✓] Stream consumed under at-most-once reading semantics.


## 🧪 6. Automated Unit & Integration Test Suite Execution
Executing all 13 unit tests verifying configuration, topic specs, schema rules, and serialization.

In [6]:
import unittest
suite = unittest.defaultTestLoader.discover('../tests')
runner = unittest.TextTestRunner(verbosity=2)
result = runner.run(suite)

test_bootstrap_servers_cluster (tests.test_pipeline.TestConfiguration.test_bootstrap_servers_cluster) ... ok
test_consumer_at_most_once_settings (tests.test_pipeline.TestConfiguration.test_consumer_at_most_once_settings) ... ok
test_producer_idempotence_settings (tests.test_pipeline.TestConfiguration.test_producer_idempotence_settings) ... ok
test_topic_partitions_and_replication (tests.test_pipeline.TestConfiguration.test_topic_partitions_and_replication) ... ok
test_distance_and_fare_ranges (tests.test_pipeline.TestDataGenerator.test_distance_and_fare_ranges) ... ok
test_driver_id_format (tests.test_pipeline.TestDataGenerator.test_driver_id_format) ... ok
test_generate_trip_schema (tests.test_pipeline.TestDataGenerator.test_generate_trip_schema) ... ok
test_passenger_id_format (tests.test_pipeline.TestDataGenerator.test_passenger_id_format) ... ok
test_pickup_and_dropoff_different (tests.test_pipeline.TestDataGenerator.test_pickup_and_dropoff_different) ... ok
test_timestamp_iso_form